## Section 1 - Introduction

This notebook demonstrates how to use the DTI Analyzer tool to explore drug-target interactions. It shows how to search for targets and drugs, view their properties, and visualize the results using free public APIs from ChEMBL and PubChem.

In [ ]:
## Section 2 - Setup

from src.chembl_query import search_target, get_bioactivities, search_molecule
from src.pubchem_query import get_compound_properties, get_compound_image_url
from src.analyzer import analyze, format_results
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
## Section 3 - Target Search Example

# Search for EGFR as a target
results = analyze("EGFR", mode="target")

# Show raw results as a pandas DataFrame
if results.get("results"):
    df = pd.DataFrame(results["results"])
    display(df.head())
else:
    print("No results found")

In [ ]:
## Section 4 - Drug Search Example

# Search for Imatinib as a drug
drug_results = analyze("Imatinib", mode="drug")

# Show compound properties
print("ChEMBL Info:", drug_results.get("chembl_info"))
print("PubChem Info:", drug_results.get("pubchem_info"))

In [ ]:
## Section 5 - Visualization

# Take the bioactivity results for EGFR
egfr_results = analyze("EGFR", mode="target")
bio_data = egfr_results.get("results", [])

if bio_data:
    # Filter for valid IC50 values
    valid_data = [d for d in bio_data if d.get("ic50_nm") is not None]
    if valid_data:
        # Limit to top 10 for plotting
        top_data = valid_data[:10]
        names = [d.get("drug_name", "Unknown")[:15] for d in top_data]
        ic50_values = [float(d.get("ic50_nm")) for d in top_data]
        
        plt.figure(figsize=(10, 6))
        plt.bar(names, ic50_values, color='skyblue')
        plt.xlabel('Molecule ChEMBL ID')
        plt.ylabel('IC50 in nM')
        plt.title('Top EGFR Inhibitors by IC50 (ChEMBL data)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()\n    else:\n        print("No valid IC50 data for plotting")\nelse:\n    print("No bioactivity results for EGFR")

## FDA Safety Data

OpenFDA is a free public database maintained by the US Food and Drug Administration that provides access to drug labels, adverse event reports, and other safety information. It helps researchers understand how drugs behave in real patients outside of controlled laboratory settings.

Drug safety data matters when studying drug-target interactions because a compound might bind strongly to a target in the lab but still cause harmful side effects in people. Combining interaction data with safety reports gives a more complete picture of whether a drug is actually useful and safe for treatment.

In [ ]:
from src.openfda_query import get_drug_label_summary
summary = get_drug_label_summary("Imatinib")
print("Drug Interactions:")
print(summary.get("interactions_text", "None"))
print("\nTop Adverse Events:")
for event in summary.get("adverse_events", []):
    print(f"  {event['reaction_name']} - Count: {event['count']}, Serious: {event['serious']}")